In [27]:
import pygad
import numpy as np
import itertools
import math

In [28]:
A = [7, 8, 18]
B = [1,2,2,4,5,5,5,9]
C = [1,2,2,2,2,3,4,5,5,7]
m = len(A)
n = len(B)
target = sorted(C)
print(f"A:{len(A)} B: {len(B)} C: {len(C)}")

A:3 B: 8 C: 10


In [29]:
# Binary encoding parameters
bits_m = math.ceil(math.log2(m))  # 2 for m=3
bits_n = math.ceil(math.log2(n))  # 3 for n=8
total_bits = m * bits_m + n * bits_n  # 30 bits
print(f"Bits per A gene: {bits_m}, Bits per B gene: {bits_n}, Total bits: {total_bits}")

Bits per A gene: 2, Bits per B gene: 3, Total bits: 30


In [31]:
# ----------------------------
# Decode binary chromosome
# ----------------------------
def decode_solution(solution):
    """Decode binary solution into mu and nu indices."""
    mu = []
    pos = 0
    for _ in range(m):
        bits = solution[pos:pos + bits_m]
        val = int(''.join(map(str, bits)), 2)
        mu.append(val)
        pos += bits_m

    nu = []
    for _ in range(n):
        bits = solution[pos:pos + bits_n]
        val = int(''.join(map(str, bits)), 2)
        nu.append(val)
        pos += bits_n

    return mu, nu

In [32]:
def cumulative_sum(arr):
    return np.cumsum(arr).tolist()

In [33]:
def step_difference(arr):
    return [arr[0]] + [arr[i] - arr[i-1] for i in range(1, len(arr))]

In [34]:
def fitness_func(ga, solution, solution_idx) -> float:
    # chromosome encoding: [ u | ν ]
    mu,nu = decode_solution(solution)



    # Check if valid permutations (exact set match)
    if set(mu) != set(range(m)) or set(nu) != set(range(n)):
        return 0.0  # invalid permutation

    A_perm = [A[i] for i in mu]
    B_perm = [B[i] for i in nu]
    C_mod = list(target)

    k = len(C_mod)
    expected_k = m + n - 1

    # --- REPAIR SEQUENCE LOGIC ---
    if k < expected_k:
        # Rule 1: Add 0-length fragments to target C
        delta = expected_k - k
        C_mod.extend([0] * delta)

    elif k > expected_k:
        # Rule 2: Add 0-length fragments to the shorter sequence among A and B
        delta = k - expected_k
        if m <= n:
            A_perm.extend([0] * delta)
        else:
            B_perm.extend([0] * delta)

    A_cum = cumulative_sum(A_perm)
    B_cum = cumulative_sum(B_perm)

    merged = sorted(A_cum + B_cum)

    steps = step_difference(merged)

    steps_sorted = sorted(set(steps))



    error = sum(abs(s - t) for s, t in zip(steps_sorted, target))
    fitness = 1.0 / (1.0 + error)
    print(f"mu: {mu}; nu:{nu} -> fitness: {fitness}")
    return fitness

In [36]:
gene_space = [0,1]

In [38]:
ga = pygad.GA(
    num_generations=1000,
    num_parents_mating=50,
    gene_type=int,
    fitness_func=fitness_func,
    sol_per_pop= 100,
    num_genes=total_bits,
    gene_space=gene_space,
    parent_selection_type="tournament",
    crossover_type="single_point",
    crossover_probability=0.75,
    mutation_type="random",
    mutation_probability=0.05,
    keep_parents=50,
    allow_duplicate_genes=True,

)

In [39]:
ga.run()

mu: [2, 0, 1]; nu:[2, 7, 0, 6, 5, 3, 4, 1] -> fitness: 0.07692307692307693
mu: [2, 0, 1]; nu:[2, 7, 0, 6, 5, 3, 1, 4] -> fitness: 0.07692307692307693
mu: [1, 2, 0]; nu:[2, 7, 0, 6, 5, 3, 1, 4] -> fitness: 0.1
mu: [1, 2, 0]; nu:[2, 7, 0, 5, 3, 1, 4, 6] -> fitness: 0.1
mu: [0, 2, 1]; nu:[2, 7, 0, 5, 3, 1, 4, 6] -> fitness: 0.125
mu: [1, 2, 0]; nu:[2, 7, 0, 5, 3, 1, 6, 4] -> fitness: 0.1
mu: [2, 0, 1]; nu:[2, 6, 4, 5, 3, 1, 7, 0] -> fitness: 0.09090909090909091
mu: [0, 2, 1]; nu:[2, 7, 4, 5, 3, 1, 0, 6] -> fitness: 0.125
mu: [2, 0, 1]; nu:[2, 7, 4, 5, 3, 1, 0, 6] -> fitness: 0.07692307692307693
mu: [0, 2, 1]; nu:[2, 3, 4, 5, 7, 1, 0, 6] -> fitness: 0.07142857142857142
mu: [2, 0, 1]; nu:[2, 0, 1, 3, 4, 7, 5, 6] -> fitness: 0.125
mu: [2, 0, 1]; nu:[2, 0, 1, 5, 4, 3, 6, 7] -> fitness: 0.08333333333333333
mu: [2, 0, 1]; nu:[6, 4, 1, 5, 3, 0, 7, 2] -> fitness: 0.1
mu: [2, 0, 1]; nu:[4, 0, 1, 3, 5, 2, 7, 6] -> fitness: 0.1
mu: [2, 0, 1]; nu:[3, 1, 5, 4, 0, 2, 6, 7] -> fitness: 0.076923076923076

In [42]:
# Optional: Print best solution
solution, solution_fitness, solution_idx = ga.best_solution()
print(f"Best fitness: {solution_fitness}")
print(f"Best solution (binary): {solution}")
mu_bin, nu_bin = decode_solution(solution)
print(f"Decoded mu: {mu_bin}, nu: {nu_bin}")

Best fitness: 0.125
Best solution (binary): [1 0 0 0 0 1 1 0 1 0 0 1 0 1 0 0 1 1 0 0 0 1 1 1 1 0 0 1 1 0]
Decoded mu: [2, 0, 1], nu: [5, 1, 2, 3, 0, 7, 4, 6]
